# GeoLife — Behavior EDA Mentor Demo

**Mục tiêu:** gom toàn bộ EDA mới sau Home/Office baseline thành một narrative duy nhất để demo với mentor.

Notebook này không scan lại raw GeoLife. Nó reuse aggregate/private caches đã được validate từ:

- 03a — behavior-first EDA;
- 03b — mobile/distributed-mobility hypothesis audit;
- 03b.1 — observation-support-controlled audit.

Câu chuyện chính:

Frozen CP1 / CP2 baseline
→ mentor challenge fixed-office / fixed-schedule assumptions
→ 03a audit population heterogeneity
→ phát hiện robust 23-user high-mobility, multiple-anchor cohort
→ 03b challenge mobile/distributed-work hypothesis
→ 03b.1 equalize observation exposure
→ final interpretation: mobility-complexity cohort, chưa phải mobile-work semantic regime.

**Privacy:** notebook chỉ hiển thị aggregate outputs, không raw identifiers, coordinates hoặc case traces.

**Production rule:** notebook này không thay đổi frozen Home/Office semantics.


## 0. Setup — reuse Modal Volume caches

Execution pattern giống các notebook EDA hiện tại:

- Modal Volume: /mnt/geolife-data
- Repo: /tmp/geolife
- Branch mặc định: eda/03b1-support-controlled-audit

Caches:

- /mnt/geolife-data/cache/03a_user_behavior_deep_dive
- /mnt/geolife-data/cache/03b_mobile_work_hypothesis
- /mnt/geolife-data/cache/03b1_observation_support_controlled

Nếu cache đã tồn tại, notebook chỉ load JSON/CSV và render aggregate tables/charts.


In [ ]:
from pathlib import Path
import json
import os
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get(
    "GEOLIFE_REPO_BRANCH",
    "eda/03b1-support-controlled-audit",
)
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
VOLUME_ROOT = Path("/mnt/geolife-data")

CACHE_03A = VOLUME_ROOT / "cache" / "03a_user_behavior_deep_dive"
CACHE_03B = VOLUME_ROOT / "cache" / "03b_mobile_work_hypothesis"
CACHE_03B1 = VOLUME_ROOT / "cache" / "03b1_observation_support_controlled"

def ensure_repo():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )

def load_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing cache: {path}. Run the corresponding audit first."
        )
    return json.loads(path.read_text(encoding="utf-8"))

def load_csv(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing cache: {path}. Run the corresponding audit first."
        )
    return pd.read_csv(path)

ensure_repo()
os.chdir(REPO_DIR)

summary_03a = load_json(CACHE_03A / "summary.json")
summary_03b = load_json(CACHE_03B / "summary.json")
summary_03b1 = load_json(CACHE_03B1 / "summary.json")

print("Repo branch:", REPO_BRANCH)
print("Loaded 03a / 03b / 03b.1 aggregate caches.")


# Part I — Frozen baseline

Trước khi làm EDA mới, upstream contract phải giữ nguyên.

Frozen CP1:

- same_second_radius_m = 10 m
- max_gap_s = 300 s
- hard_speed_guard_kmh = 1200 km/h
- stay distance threshold = 200 m
- minimum dwell = 1200 s

Full release đã reconcile:

- 182 release users;
- 136 users có ít nhất một CP1 stay;
- 5,821 stays;
- frozen CP2 v1 comparator emit 27 HOME và 16 OFFICE.

**Lý do freeze:** nếu vừa thay cleaning/stay detection vừa làm behavior EDA, downstream difference có thể chỉ là artifact của upstream change.


In [ ]:
recon = summary_03a["reconciliation"]
emissions = summary_03a["comparator_emissions"]

display(
    pd.DataFrame(
        {
            "metric": [
                "Release users",
                "Users with CP1 stays",
                "Detected CP1 stays",
                "Frozen HOME emitted",
                "Frozen OFFICE emitted",
            ],
            "value": [
                recon["release_users"],
                recon["stay_users"],
                recon["stays"],
                emissions["HOME"],
                emissions["OFFICE"],
            ],
        }
    )
)


# Part II — 03a Behavior-first EDA

## 1. Mentor challenge

Home/Office baseline ban đầu dùng fixed behavioral windows:

- HOME: night dwell;
- OFFICE: weekday 09–17 dwell.

Mentor challenge:

- không phải ai cũng có fixed office;
- user có thể có nhiều recurring locations;
- schedule có thể khác 09–17;
- outlier có thể là valid behavior;
- work regime có thể ảnh hưởng mobility.

Thay vì sửa heuristic ngay, 03a hỏi trước:

**Behavior của user trong GeoLife thực sự heterogeneous đến mức nào?**


## 2. Coverage funnel — abstention có thể chỉ là thiếu support

Không emit label không đồng nghĩa behavior lạ.

03a tách:

release user
→ có CP1 stay hay không
→ có đủ schedule support hay không.

Đây là bước cần thiết trước khi diễn giải behavioral pattern.


In [ ]:
coverage = summary_03a["coverage"]

funnel = pd.DataFrame(
    {
        "stage": ["Release", "CP1 stay users", "Schedule-supported"],
        "users": [
            coverage["release_users"],
            coverage["stay_users"],
            coverage["schedule_supported_users"],
        ],
    }
)
display(funnel)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(funnel["stage"], funnel["users"])
ax.set_ylabel("Users")
ax.set_title("03a — Evidence coverage funnel")
plt.tight_layout()
plt.show()


Measured result:

- 182 release users;
- 136 users có CP1 stays;
- 46 schedule-supported;
- 90 stay-users vẫn chưa đủ schedule support.

**Không được suy ra:** 90 users đó có schedule bất thường. Chỉ biết evidence hiện tại chưa đủ mạnh.


## 3. Spatial heterogeneity — fixed-anchor assumption có đủ không?

03a audit recurring-anchor structure tại 200 m để tránh assume rằng user chủ yếu chỉ có Home + Office.


In [ ]:
anchors = summary_03a["anchors"]

anchor_table = pd.DataFrame(
    {
        "anchor_class": [
            "Dominant",
            "Two-anchor",
            "Multiple-anchor",
            "No stable anchor",
        ],
        "users": [
            anchors["dominant_anchor"],
            anchors["two_anchor"],
            anchors["multiple_anchor"],
            anchors["no_stable_anchor"],
        ],
    }
)
display(anchor_table)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(anchor_table["anchor_class"], anchor_table["users"])
ax.set_ylabel("Users with CP1 stays")
ax.set_title("03a — Recurring-anchor heterogeneity at 200 m")
plt.tight_layout()
plt.show()


Kết quả trên 136 stay-users:

- 19 dominant-anchor;
- 13 two-anchor;
- 72 multiple-anchor;
- 32 no-stable-anchor.

**Decision:** spatial behavior đa dạng hơn fixed Home+Office view; semantic layer không nên assume mọi user chỉ cần hai anchors.


## 4. Schedule personalization — vì sao chưa dùng JSD để thay fixed windows?

Một redesign tự nhiên là personalize work schedule theo user.

Nhưng trước khi làm vậy, cần chứng minh within-user schedule ổn định hơn null/reference.


In [ ]:
cal = summary_03a["schedule_calibration"]

display(
    pd.DataFrame(
        {
            "comparison": [
                "Within-user weekly JSD",
                "Between-user JSD",
                "Shuffled-week null",
            ],
            "median": [
                cal["within_user"]["median"],
                cal["between_user"]["median"],
                cal["shuffled_week"]["median"],
            ],
            "q1": [
                cal["within_user"]["q1"],
                cal["between_user"]["q1"],
                cal["shuffled_week"]["q1"],
            ],
            "q3": [
                cal["within_user"]["q3"],
                cal["between_user"]["q3"],
                cal["shuffled_week"]["q3"],
            ],
        }
    )
)


Measured result:

- within-user weekly JSD median = 1.000;
- between-user median = 0.904;
- shuffled-week null = 1.000.

**Decision:** current JSD check không establish personalized schedule structure.

Vì vậy không dùng JSD để personalize 09–17, không tạo user-specific OFFICE window, và không dùng nó làm v2 cutoff.

Đây là negative result quan trọng: EDA loại bỏ một redesign chưa đủ evidence.


## 5. Mobility hypothesis xuất hiện từ đâu?

03a đo cleaned-point mobility trên usable temporal-profile days:

- distance/day;
- movement-duration proxy/day;
- boundary count/day.

Đây là movement proxies, không phải true travel distance.

Trong audit xuất hiện một cohort 23 users:

- 23/23 multiple-anchor;
- 23/23 frozen OFFICE-abstained;
- 0/23 OFFICE-emitted;
- chỉ 2/23 có repeatable exact motif.

Tên mobile-work-like ở đây chỉ là descriptive wrapper, chưa phải occupation inference.


In [ ]:
mobility = summary_03a["mobility"]
overlap = summary_03a["candidate_overlap"]["mobile_work_like_intersections"]
counts = summary_03a["candidate_overlap"]["dimension_counts"]

display(
    pd.DataFrame(
        {
            "metric": [
                "Cleaned distance (km/day)",
                "Movement-duration proxy (h/day)",
                "Boundary count (/day)",
            ],
            "median": [
                mobility["distance_per_usable_day_km"]["median"],
                mobility["movement_proxy_per_usable_day_h"]["median"],
                mobility["boundary_count_per_usable_day"]["median"],
            ],
            "q1": [
                mobility["distance_per_usable_day_km"]["q1"],
                mobility["movement_proxy_per_usable_day_h"]["q1"],
                mobility["boundary_count_per_usable_day"]["q1"],
            ],
            "q3": [
                mobility["distance_per_usable_day_km"]["q3"],
                mobility["movement_proxy_per_usable_day_h"]["q3"],
                mobility["boundary_count_per_usable_day"]["q3"],
            ],
        }
    ).round(2)
)

display(
    pd.DataFrame(
        {
            "measure": [
                "Mobile-work-like candidates",
                "Multiple-anchor",
                "Repeatable exact motif",
                "Frozen OFFICE emitted",
                "Frozen OFFICE abstained",
            ],
            "users": [
                counts["mobile_work_like"],
                overlap["multiple_anchor"],
                overlap["motif_repeatable"],
                overlap["OFFICE_emitted"],
                overlap["OFFICE_abstained"],
            ],
        }
    )
)


# Part III — 03b Hypothesis audit

Question:

**Nhóm 23 users có phải distinct distributed/mobile-work-like behavioral regime hay chỉ là artifact của wrapper, support hoặc travel?**

Thiết kế:

- Group A = frozen 23 candidates;
- Group B = matched OFFICE-abstained controls;
- Group C = frozen OFFICE-emitted comparator.

Anti-circularity rule:

Không dùng lại chính construction features để validate hypothesis.

Independent evidence ưu tiên:

- route/transition structure;
- transport modes khi có labels;
- sensitivity;
- negative controls.


In [ ]:
groups = summary_03b["groups"]
display(
    pd.DataFrame(
        {
            "group": ["A candidate", "B matched control", "C OFFICE comparator"],
            "users": [groups["A"], groups["B"], groups["C"]],
        }
    )
)

sens = load_csv(CACHE_03B / "sensitivity.csv")
display(sens)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(sens["variant"], sens["jaccard"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Jaccard vs frozen 23")
ax.set_title("03b — Candidate robustness")
ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()


Sensitivity result:

- anchor 100 m → same 23;
- anchor 300 m → same 23;
- mobility threshold ±10% chỉ đổi khoảng 1 user;
- support ±1 weekday không đổi cohort.

**Decision:** cohort robust với tested thresholds.

Nhưng robust definition không đồng nghĩa validated semantics.


## 6. Matching confound

03b match đủ A/B pairs nhưng aggregate observation support vẫn lệch.

Nếu A được observe nhiều hơn, ta có thể thấy nhiều transitions/entropy hơn chỉ vì coverage.

Vì vậy route difference ở 03b chưa được đọc như causal/independent evidence.


In [ ]:
support = load_csv(CACHE_03B / "support_balance.csv")
support_groups = support[support["group"].isin(["A", "B", "C"])].copy()

display(
    support_groups[
        [
            "group",
            "active_days_median",
            "usable_temporal_days_median",
            "cp1_stay_count_median",
        ]
    ]
)

route_03b = pd.DataFrame(
    [{"group": g, **values} for g, values in summary_03b["transition_summary"].items()]
)
display(route_03b.round(3))

mode_03b = pd.DataFrame(summary_03b["transport_mode"])
display(
    mode_03b[
        [
            "group",
            "label_users",
            "matched_users",
            "median_matched_duration_h_per_user",
            "median_matched_distance_km_per_user",
            "motorized_mode_distance_share",
        ]
    ].round(3)
)


03b result:

- candidate robustness: strong;
- route complexity difference: suggestive;
- recurrent route difference: weak;
- transport coverage: sparse;
- A/B observation balance: problematic.

Research decision: **mixed evidence**.

Đây là lý do cần 03b.1.


# Part IV — 03b.1 Observation-support-controlled audit

Câu hỏi mới:

**Nếu cho A và B cùng lượng usable observation exposure, khác biệt còn không?**

Với mỗi matched pair:

- downsample weekday A/B về cùng count;
- downsample weekend A/B về cùng count;
- route analysis làm riêng trên usable_for_motif days;
- bootstrap 500 lần;
- transport subset control theo matched labeled hours.

Sampling chỉ thay exposure, không thay frozen candidate definition.


In [ ]:
print("03b.1 support:", summary_03b1["support"])
print("Bootstrap repetitions:", summary_03b1["n_bootstraps"])

mobility_03b1 = load_csv(CACHE_03B1 / "mobility_summary.csv")
display(mobility_03b1.round(4))

x = np.arange(len(mobility_03b1))
y = mobility_03b1["paired_difference_median"].to_numpy()
low = y - mobility_03b1["ci95_low"].to_numpy()
high = mobility_03b1["ci95_high"].to_numpy() - y

fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar(x, y, yerr=np.vstack([low, high]), fmt="o", capsize=4)
ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(mobility_03b1["metric"], rotation=45, ha="right")
ax.set_ylabel("Paired A - B difference")
ax.set_title("03b.1 — Mobility after equalizing exposure")
plt.tight_layout()
plt.show()


Mobility after exposure control:

- cleaned distance/day ≈ +22.88 km for A;
- movement proxy/day ≈ +0.74 h;
- boundary count/day ≈ +0.88;
- recurring-location count/day ≈ no difference;
- stay count/day ≈ no clear difference.

**Interpretation:** richer observation không giải thích hết movement magnitude.

Nhưng mobility metrics overlap với candidate construction, nên chưa phải independent semantic validation.


## 7. Independent route evidence sau exposure control

Đây là evidence stream quan trọng nhất.

Nếu route structure vẫn khác sau khi equalize days, difference không chỉ do A được observe nhiều hơn.


In [ ]:
route_03b1 = load_csv(CACHE_03B1 / "route_summary.csv")
display(route_03b1.round(4))

x = np.arange(len(route_03b1))
y = route_03b1["paired_difference_median"].to_numpy()
low = y - route_03b1["ci95_low"].to_numpy()
high = route_03b1["ci95_high"].to_numpy() - y

fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar(x, y, yerr=np.vstack([low, high]), fmt="o", capsize=4)
ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(route_03b1["metric"], rotation=45, ha="right")
ax.set_ylabel("Paired A - B difference")
ax.set_title("03b.1 — Route evidence after equalizing exposure")
plt.tight_layout()
plt.show()

transport_03b1 = load_csv(CACHE_03B1 / "transport_summary.csv")
display(transport_03b1.round(4))
print("Eligible transport pairs:", summary_03b1["support"]["transport_pairs_eligible"])


Route result:

- edge entropy A-B ≈ +0.232;
- 95% bootstrap interval ≈ [0.006, 0.455];
- recurrent edges/day difference = 0;
- transition/day interval touches 0;
- distinct-edge/day interval touches 0.

**What survives:** route diversity/complexity.

**What does not survive as strong evidence:** recurrent route structure.

Transportation evidence chỉ có 1 eligible matched pair, nên không dùng cho population-level conclusion.


# Part V — Final research decision

| Stage | Question | Result | Decision |
|---|---|---|---|
| 03a | Behavior heterogeneous? | 72/136 multiple-anchor | Không assume Home+Office đủ cho mọi user |
| 03a schedule | Personalized schedule có evidence? | JSD không distinguish rõ khỏi null | Không personalize windows bằng JSD |
| 03a candidate | Có high-mobility OFFICE-abstained cohort? | 23 users | Tạo hypothesis, chưa semanticize |
| 03b | Cohort robust + independent evidence? | Robust, nhưng mixed + support imbalance | mixed evidence |
| 03b.1 | Support imbalance giải thích hết? | Không; movement + edge entropy còn | Freeze như mobility-complexity cohort |

Final interpretation:

23-user cohort
→ robust under threshold perturbation
→ not explained only by richer observation coverage
→ higher movement magnitude + route diversity
→ BUT no stronger recurrent-route evidence
→ transport labels too sparse
→ no occupation/work ground truth
→ robust descriptive mobility-complexity cohort
→ NOT validated mobile-work semantic regime.


## Production decision

Không thay đổi frozen Home/Office semantics từ 03a/03b/03b.1.

Giữ nguyên:

- CP1 cleaning/stay detection;
- CP2 v1 comparator;
- Home/Office production.

Lý do:

- chưa có Home/Office/work-role ground truth;
- recurrent route evidence chưa support hypothesis;
- transportation labels quá sparse;
- semantic leap từ mobility sang occupation chưa được validate.

Nếu sau này có external labels, 23-user cohort là candidate set tốt để validate future regime classifier.


# Mentor-ready talking points

### Nếu mentor hỏi: Em đã làm gì sau feedback?

Em không sửa heuristic ngay. Em tách feedback thành các hypothesis có thể falsify. Đầu tiên em audit heterogeneity toàn population. Sau đó em thấy một cohort 23 users nhiều anchors và mobility cao mà frozen OFFICE abstain toàn bộ. Em kiểm tra cohort qua sensitivity, matched controls, route structure và transport labels. Vì matched controls vẫn lệch observation support, em làm thêm pairwise exposure-controlled bootstrap. Sau control, movement magnitude và route diversity vẫn còn, nhưng recurrent route evidence và transport coverage không đủ. Vì vậy em giữ finding ở mức mobility-complexity cohort, chưa gọi là mobile-work và chưa sửa production semantics.

### Nếu mentor hỏi: Feedback mobile worker có đúng không?

Feedback đúng như một challenge đối với fixed-office assumption, nhưng dataset hiện tại chưa đủ evidence để xác nhận occupation/mobile-work semantics.

### Nếu mentor hỏi: Tại sao không tune thêm?

Vì tune thêm threshold trên cùng dataset dễ thành overfitting. Semanticization cần independent labels/evaluation, không chỉ thêm heuristic.


# Open follow-up — DBSCAN MinPts

Mentor note còn lại:

DBSCAN hiện dùng min_samples=1. Eps đã được sensitivity nhưng MinPts chưa được justify.

Với min_samples=1:

- mọi point tự nó là core;
- isolated stays trở thành singleton clusters;
- density-based noise gần như không tồn tại;
- connectivity/chaining permissive hơn.

Follow-up experiment:

- min_samples = 1 / 2 / 3 / 5;
- kết hợp representative eps values;
- đo recurring locations/users, noise rate, singleton behavior, p95/max diameter, clusters >200 m, users mất recurring-location support;
- compare với complete-link 200 m.

Decision rule: không chọn MinPts chỉ vì coverage cao nhất; cần explainable trade-off giữa coverage, density requirement, compactness và downstream support.


# Final takeaway

**EDA mới không chứng minh mobile workers. Nó chứng minh current fixed-office view không mô tả hết behavioral diversity, và cô lập được một robust mobility-complexity cohort đáng giữ lại cho future validation.**

Research insight → giữ.

Semantic label → chưa.

Production change → chưa.
